<a href="https://colab.research.google.com/github/riyapai05/TULU_PROJECT/blob/main/Tulu_Font_Identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Tulu dataset.zip to Tulu dataset.zip


In [ ]:
import zipfile

with zipfile.ZipFile(
    "Tulu dataset.zip",
    "r"
) as zip_ref:

    zip_ref.extractall(
        "/content/tulu_dataset"
    )

print("Dataset Extracted")

Dataset Extracted


In [ ]:
import os

for root, dirs, files in os.walk("/content/tulu_dataset"):
    print(root)

/content/tulu_dataset
/content/tulu_dataset/Tulu dataset
/content/tulu_dataset/Tulu dataset/ಇ
/content/tulu_dataset/Tulu dataset/ಅ
/content/tulu_dataset/Tulu dataset/ಓ
/content/tulu_dataset/Tulu dataset/ಈ
/content/tulu_dataset/Tulu dataset/ಊ
/content/tulu_dataset/Tulu dataset/ಎ
/content/tulu_dataset/Tulu dataset/ಏ
/content/tulu_dataset/Tulu dataset/ಉ
/content/tulu_dataset/Tulu dataset/ಐ
/content/tulu_dataset/Tulu dataset/ಔ
/content/tulu_dataset/Tulu dataset/ಅಂ
/content/tulu_dataset/Tulu dataset/ಉ್
/content/tulu_dataset/Tulu dataset/ಋ
/content/tulu_dataset/Tulu dataset/ಒ
/content/tulu_dataset/Tulu dataset/ಅಃ
/content/tulu_dataset/Tulu dataset/ಆ
/content/tulu_dataset/Tulu dataset/ೠ


In [ ]:
import os
import shutil
import random

SOURCE = "/content/tulu_dataset/Tulu dataset"

TRAIN_DIR = "/content/train_dataset"
TEST_DIR = "/content/test_dataset"

# Delete old folders if they exist
if os.path.exists(TRAIN_DIR):
    shutil.rmtree(TRAIN_DIR)

if os.path.exists(TEST_DIR):
    shutil.rmtree(TEST_DIR)

# Create fresh folders
os.makedirs(TRAIN_DIR)
os.makedirs(TEST_DIR)

for cls in os.listdir(SOURCE):

    cls_path = os.path.join(SOURCE, cls)

    if not os.path.isdir(cls_path):
        continue

    train_cls = os.path.join(TRAIN_DIR, cls)
    test_cls = os.path.join(TEST_DIR, cls)

    os.makedirs(train_cls)
    os.makedirs(test_cls)

    images = os.listdir(cls_path)

    random.shuffle(images)

    split = int(0.8 * len(images))

    train_images = images[:split]
    test_images = images[split:]

    # Copy training images
    for img in train_images:

        shutil.copy(
            os.path.join(cls_path, img),
            os.path.join(train_cls, img)
        )

    # Copy testing images
    for img in test_images:

        shutil.copy(
            os.path.join(cls_path, img),
            os.path.join(test_cls, img)
        )

print("Split Completed")

Split Completed


In [ ]:
import os

train_count = 0
test_count = 0

for cls in os.listdir("/content/train_dataset"):

    train_count += len(
        os.listdir(
            os.path.join(
                "/content/train_dataset",
                cls
            )
        )
    )

for cls in os.listdir("/content/test_dataset"):

    test_count += len(
        os.listdir(
            os.path.join(
                "/content/test_dataset",
                cls
            )
        )
    )

print("Train Images:", train_count)
print("Test Images:", test_count)

Train Images: 136
Test Images: 34


In [ ]:
from PIL import Image
from torchvision import transforms
import os

TRAIN_DIR = "/content/train_dataset"

augment = transforms.Compose([

    transforms.RandomRotation(15),

    transforms.RandomAffine(
        degrees=0,
        translate=(0.1,0.1),
        scale=(0.9,1.1)
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    )
])

for cls in os.listdir(TRAIN_DIR):

    cls_path = os.path.join(
        TRAIN_DIR,
        cls
    )

    originals = os.listdir(cls_path)

    for img_name in originals:

        img = Image.open(
            os.path.join(
                cls_path,
                img_name
            )
        ).convert("RGB")

        for i in range(20):

            aug_img = augment(img)

            aug_img.save(
                os.path.join(
                    cls_path,
                    f"aug_{i}_{img_name}"
                )
            )

print("Augmentation Done")

Augmentation Done


In [ ]:
!pip install timm -q

In [ ]:
import torch
import torch.nn as nn
import timm

from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
train_dataset = ImageFolder(
    root="/content/train_dataset",
    transform=transform
)

test_dataset = ImageFolder(
    root="/content/test_dataset",
    transform=transform
)

num_classes = len(train_dataset.classes)

print("Classes:", num_classes)
print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

Classes: 17
Train: 2856
Test: 34


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [ ]:
model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=num_classes,
    drop_rate=0.4
)

model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [ ]:
epochs = 10

for epoch in range(epochs):

    model.train()

    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        _, predicted = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

    print(
        f"Epoch {epoch+1}: "
        f"{100*correct/total:.2f}%"
    )

Epoch 1: 47.34%
Epoch 2: 93.56%
Epoch 3: 98.49%
Epoch 4: 99.40%
Epoch 5: 99.68%
Epoch 6: 99.79%
Epoch 7: 99.65%
Epoch 8: 99.86%
Epoch 9: 99.82%
Epoch 10: 99.93%


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

print(
    "Test Accuracy:",
    100 * correct / total
)

Test Accuracy: 91.17647058823529


In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Upload image
uploaded = files.upload()

# Get filename
img_path = list(uploaded.keys())[0]

# Open image
img = Image.open(img_path).convert("RGB")

# Display image
plt.figure(figsize=(5,5))
plt.imshow(img)
plt.axis("off")
plt.show()

# Transform image
img_tensor = transform(img).unsqueeze(0).to(device)

# Prediction
model.eval()

with torch.no_grad():

    output = model(img_tensor)

    probabilities = F.softmax(output, dim=1)

    confidence, predicted = torch.max(probabilities, 1)

predicted_class = train_dataset.classes[predicted.item()]

print(f"Predicted Class : {predicted_class}")
print(f"Confidence      : {confidence.item()*100:.2f}%")